# N3 Slinky Sim Architecture Sweep

Runs all architecture variants on the 3-node simulated slinky data using the same core model/training settings as `2d_slinky_train_from_sim_LLT.ipynb`. All outputs are collected under one `OUTPUT_DIR`, with per-architecture plots, energy landscapes, paper-ready comparison plots, and post-training Hessian diagnostics.

In [1]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import SlinkyN3Properties
from run_architectures import SweepConfig, run_architecture_sweep

# ---------------------------------------------------------
# Dataset/properties/model/training settings copied from
# 2d_slinky_train_from_sim_LLT.ipynb.
# ---------------------------------------------------------
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2"
PAPER_PLOT_DIR = OUTPUT_DIR / "paper_ready_architecture_comparison"

train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset_7_trajs.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset_6_trajs.npz"

properties = SlinkyN3Properties()
K_init_chol = (0.02, 0.0, 0.05)
K_init_diag = (0.02, 0.05)

cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10, 10),
    corr_factor=0.05,
    input_mode="invariant",
    only_stretching_NN=True,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    n_epochs=500,
    lr=1e-3,
    seed=42,
    valid_every=1,
    max_dlambda=1e-2,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-4,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key="F",
    force_loss_strength=0.1,
    force_components=(0,),
    force_sign=1.0,
    return_loss_components=True,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    early_stopping_warmup_epochs=200,
    restore_best_model=True,
    output_dir=str(OUTPUT_DIR),
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=True,
    plot_force_predictions=True,
    save_hessian_diagnostics=False,  # post-process in the Hessian cell below
    save_energy_landscapes=True,
    energy_snapshot_initial=True,
    energy_snapshot_final=True,
    energy_snapshot_epochs=(),
    energy_snapshot_every=None,
    energy_snapshot_use_valid=True,
    energy_snapshot_dpi=180,
    energy_snapshot_n_grid=None,
    verbose=True,
    continue_on_failure=True,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving all results under: {OUTPUT_DIR.resolve()}")

Saving all results under: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2


In [2]:
from run_architectures import subset_all

# # All registered architecture variants.
# selected_architectures = subset_all()

selected_architectures = [
    "diag_stiffness_icnn",
    "chol_stiffness_icnn",
]

# selected_architectures = [
#     arch for arch in selected_architectures
#     if arch not in ["chol_stiffness_signed_mlp", "chol_stiffness_signed_icnn"]
# ]



print(f"Running {len(selected_architectures)} architectures:")
for name in selected_architectures:
    print(f"  - {name}")

print("\nPer-architecture plots:", cfg.save_plots)
print("Energy landscape snapshots:", cfg.save_energy_landscapes)
print("Force prediction plots:", cfg.plot_force_predictions)

Running 2 architectures:
  - diag_stiffness_icnn
  - chol_stiffness_icnn

Per-architecture plots: True
Energy landscape snapshots: True
Force prediction plots: True


In [3]:
results = run_architecture_sweep(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=cfg,
    selected_architectures=selected_architectures,
)

successes = [name for name, result in results.items() if result["success"]]
failures = {name: result["failure_reason"] for name, result in results.items() if not result["success"]}

print(f"Succeeded: {len(successes)}/{len(results)}")
if failures:
    print("Failures:")
    for name, reason in failures.items():
        print(f"  - {name}: {reason}")
else:
    print("No architecture failures.")

Running architecture: diag_stiffness_icnn
  model_cls               : DiagonalPlusStiffnessNN
  which_case              : ICNN
  hidden                  : (10, 10)
  input_mode              : invariant
  only_stretching_NN      : True
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.05
  zero_reference          : True
  seed                    : 42
  n_epochs                : 500
  lr                      : 0.001
  weight_decay            : 0.0
  max_dlambda             : 0.01
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  early_stopping_warmup   : 200
  restore_best_model      : True
  hessian_reg_strength    : 0.0

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Running architecture: chol_stiffness_icnn
  model_cls               : CholeskyPlusStiffnessNN
  which_case              : ICNN
  hidden                  : (10, 10)
  input_mode              : invariant
  only_stretching_NN      : True
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.05
  zero_reference          : True
  seed                    : 42
  n_epochs                : 500
  lr                      : 0.001
  weight_decay            : 0.0
  max_dlambda             : 0.01
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  early_stopping_warmup   : 200
  restore_best_model      : True
  hessian_reg_strength    : 0.0

In [4]:
from architecture_plots import plot_architecture_comparison_paper, plot_summary_final_losses

PAPER_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Paper-ready PDF comparisons across all successfully completed architectures.
paper_architectures = successes if "successes" in globals() else selected_architectures
paper_paths = plot_architecture_comparison_paper(
    architectures=paper_architectures,
    results_dir=str(OUTPUT_DIR),
    output_dir=str(PAPER_PLOT_DIR),
    traj_idx=0,
)

# Convenience PNG summary of final losses from the in-memory results dict.
plot_summary_final_losses(
    {name: results[name] for name in paper_architectures},
    save_path=str(PAPER_PLOT_DIR / "final_loss_summary.png"),
    show=False,
)

print("Paper-ready plots written to:", PAPER_PLOT_DIR.resolve())
for key, path in paper_paths.items():
    if key != "colors":
        print(f"  {key}: {path}")

Paper-ready plots written to: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2/paper_ready_architecture_comparison
  training_loss: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2/paper_ready_architecture_comparison/training_loss_comparison.pdf
  validation_loss: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2/paper_ready_architecture_comparison/validation_loss_comparison.pdf
  training_trajectory: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2/paper_ready_architecture_comparison/training_trajectory_comparison.pdf
  validation_trajectory: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sw

In [5]:
import subprocess
import sys

# Post-training Hessian diagnostics. This is intentionally separated from
# training so the architecture sweep stays fast.
HESSIAN_USE_PREDICTED = True
HESSIAN_STRIDE = 10
HESSIAN_MAX_TRAJECTORIES = 1  # set to None to process all trajectories
HESSIAN_SPLITS = ("train", "valid")

cmd = [
    sys.executable,
    "compute_architecture_hessian_diagnostics.py",
    str(OUTPUT_DIR),
    "--stride",
    str(HESSIAN_STRIDE),
    "--splits",
    *HESSIAN_SPLITS,
]
if HESSIAN_USE_PREDICTED:
    cmd.append("--use-predicted")
if HESSIAN_MAX_TRAJECTORIES is None:
    cmd.append("--all-trajectories")
else:
    cmd.extend(["--max-trajectories", str(HESSIAN_MAX_TRAJECTORIES)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)
print("Hessian diagnostics written into each architecture directory under:", OUTPUT_DIR.resolve())

Running: /Users/radha/GitRepos/dismech-jax/.venv/bin/python compute_architecture_hessian_diagnostics.py /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2 --stride 10 --splits train valid --use-predicted --max-trajectories 1
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2/chol_stiffness_icnn__hid_10x10__inp_invariant__stretchNN_1__bendNN_0__act_tanh__corr_0.05__zr1__seed_42__mdl_0.01__it_20__hreg_0.0001__hprobe_1__hseed_0__floss_0.1__fcomp_0__fsign_1 | train: M=1.499e+00, kappa=2.169e+01, states=15 | valid: M=1.887e+00, kappa=2.853e+01, states=15
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2/diag_stiffness_icnn__hid_10x10__inp_invariant__stretchNN_1__bendNN_0__act_tanh__corr_0.05__zr1__seed_

In [6]:
print("All artifacts are organized under:")
print(" ", OUTPUT_DIR.resolve())
print("\nMain subfolders/files to inspect:")
print("  - <architecture>/results.npz")
print("  - <architecture>/model.eqx")
print("  - <architecture>/loss_curves.png")
print("  - <architecture>/pred_vs_truth_*")
print("  - <architecture>/force_pred_vs_truth_*.png")
print("  - <architecture>/energy_landscapes/")
print("  - <architecture>/hessian_diagnostics_*.npz")
print("  - <architecture>/hessian_diagnostics_summary.json")
print("  - paper_ready_architecture_comparison/*.pdf")

All artifacts are organized under:
  /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n3_slinky_sim_all_architectures_fail_on_nonconvergence_False_run2

Main subfolders/files to inspect:
  - <architecture>/results.npz
  - <architecture>/model.eqx
  - <architecture>/loss_curves.png
  - <architecture>/pred_vs_truth_*
  - <architecture>/force_pred_vs_truth_*.png
  - <architecture>/energy_landscapes/
  - <architecture>/hessian_diagnostics_*.npz
  - <architecture>/hessian_diagnostics_summary.json
  - paper_ready_architecture_comparison/*.pdf
